# Chain-of-Thought Prompting with LLaMA-3.2

**Course:** Natural Language Processing · Unit 4 — Prompt Engineering  
**Institution:** Universidad Tecnológica de Bolívar  
**Reference:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3rd ed.), Ch. 12 — Prompting and In-Context Learning  
**Python compatibility:** 3.10 · 3.11 · 3.12 · 3.13

---

## Learning Objectives

1. Load a local instruction-tuned LLM (LLaMA-3.2-1B-Instruct) with HuggingFace Pipelines
2. Contrast standard prompting against Chain-of-Thought (CoT) prompting
3. Extract only the assistant turn from a chat-generation output
4. Observe qualitative differences in model reasoning quality


---

## 1. Environment Setup

Install once in your terminal:

```bash
pip install transformers torch accelerate
```


In [ ]:
# pip install transformers torch accelerate  # run once in terminal
from html import unescape
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

import transformers
print(transformers.__version__)


> **Output interpretation:** Confirms the Transformers library is installed. Version 4.40+ is required for LLaMA-3.2 chat templates.


---

## 2. Load the Model

We use `meta-llama/Llama-3.2-1B-Instruct`, the 1-billion-parameter instruct variant.  
With `device_map='auto'`, the model loads onto GPU if available, CPU otherwise.


In [ ]:
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

pipe = pipeline(
    "text-generation",
    model=MODEL_ID,
    device_map="auto",
)
print(f"Pipeline ready: {MODEL_ID}")


> **Output interpretation:** Loading a 1B-parameter model takes roughly 2 GB of RAM/VRAM and 30-60 seconds on first run (the weights are downloaded and cached). On CPU-only machines, inference will be slow (~1 min/response).


---

## 3. Standard Prompt (No Chain-of-Thought)

The base prompt gives the model a question with no reasoning instructions.  
Small models often skip intermediate steps and guess the answer directly.


In [ ]:
prompt_no_cot = [
    {
        "role": "user",
        "content": "If Sarah has 3 apples and gives half to Tom, how many does she have left?",
    }
]

outputs = pipe(prompt_no_cot, max_new_tokens=200)

# Extract only the assistant's reply from the generated conversation
response_no_cot = next(
    item["content"] for item in outputs[0]["generated_text"]
    if item["role"] == "assistant"
)
print("=== Without CoT ===")
print(response_no_cot)


> **Output interpretation:** Without a CoT trigger, smaller models frequently answer '1.5' or give a direct numeric answer with no explanation. Observe whether the model justifies its reasoning or just states a number.


---

## 4. Chain-of-Thought Prompt

Adding **'Let's think step by step.'** to the question prompts the model to  
decompose the problem before committing to an answer — a technique that  
dramatically improves accuracy on arithmetic and commonsense reasoning tasks  
(Wei et al., 2022).


In [ ]:
prompt_with_cot = [
    {
        "role": "user",
        "content": (
            "If Sarah has 3 apples and gives half to Tom, how many does she have left? "
            "Let's think step by step."
        ),
    }
]

outputs = pipe(prompt_with_cot, max_new_tokens=300)

response_with_cot = next(
    item["content"] for item in outputs[0]["generated_text"]
    if item["role"] == "assistant"
)
print("=== With CoT ===")
print(response_with_cot)


> **Output interpretation:** With the CoT trigger, the model should enumerate steps: (1) half of 3 is 1.5, (2) Sarah keeps 1.5 apples. The answer is the same, but the intermediate reasoning is now visible, making it easier to verify correctness and catch errors.


---

## Summary

| Prompt type | Token overhead | Accuracy lift | Best for |
|---|---|---|---|
| Standard zero-shot | None | Baseline | Simple factual Q&A |
| Zero-shot CoT | ~5 trigger tokens | +10–40 pp on math | Arithmetic, logic, multi-step reasoning |
| Few-shot CoT | Exemplar tokens | Highest | Tasks with a fixed answer format |

CoT prompting is most effective with models > 100B parameters. Smaller models like LLaMA-3.2-1B may show modest improvement; larger variants (7B+) show stronger gains.
